In [ ]:
import requests

import pandas as pd

import msal

from pyspark.sql import SparkSession
 
# ── Config ──────────────────────────────────────────────────────────────

tenant_id   = "9a5cacd0-2bef-4dd7-ac5c-7ebe1f54f495"

client_id   = "51f81489-12ee-4a9e-aaae-a2591f45987d"  # Well-known Power BI client ID

org_url     = "https://esacontact.crm4.dynamics.com"

entity      = "cap_cap_esacontactonboarding_systemusers"

table_name  = "cap_esacontact_systemusers"   # Delta table name in Lakehouse
 
# ── Auth (device code flow) ──────────────────────────────────────────────

app = msal.PublicClientApplication(

    client_id,

    authority=f"https://login.microsoftonline.com/{tenant_id}"

)
 
accounts = app.get_accounts()

result = app.acquire_token_silent([f"{org_url}/.default"], account=accounts[0]) if accounts else None
 
if not result:

    flow = app.initiate_device_flow(scopes=[f"{org_url}/.default"])

    print(flow["message"])  # ← go to microsoft.com/devicelogin and enter this code

    result = app.acquire_token_by_device_flow(flow)
 
if "access_token" not in result:

    raise Exception(f"Auth failed: {result.get('error_description')}")
 
token = result["access_token"]

print("✅ Token acquired")
 
# ── Pull data from Dataverse OData ──────────────────────────────────────

headers = {

    "Authorization": f"Bearer {token}",

    "OData-MaxVersion": "4.0",

    "OData-Version": "4.0",

    "Accept": "application/json",

    "Prefer": "odata.maxpagesize=5000"

}
 
all_records = []

url = f"{org_url}/api/data/v9.2/{entity}"
 
while url:

    resp = requests.get(url, headers=headers)

    resp.raise_for_status()

    data = resp.json()

    all_records.extend(data.get("value", []))

    url = data.get("@odata.nextLink")

    print(f"  → fetched {len(all_records)} records so far...")
 
print(f"✅ Total records pulled: {len(all_records)}")
 
# ── Pandas → Spark → Delta ───────────────────────────────────────────────

pdf = pd.DataFrame(all_records)
 
# Clean column names (Dataverse prefixes like @odata.etag cause issues)

pdf.columns = [c.replace("@", "_").replace(".", "_") for c in pdf.columns]
 
spark = SparkSession.builder.getOrCreate()

sdf = spark.createDataFrame(pdf)
 
# Write as Delta table into the attached Lakehouse

(

    sdf.write

    .format("delta")

    .mode("overwrite")          # change to "append" if you want incremental

    .option("overwriteSchema", "true")

    .saveAsTable(table_name)

)
 
print(f"✅ Written to Delta table: {table_name}")

display(spark.sql(f"SELECT * FROM {table_name} LIMIT 10"))
 
